In [ ]:
import pandas as pd

df = pd.read_csv("olist_2016_2018.csv")

display(df)
display(df.info())

In [ ]:
# Gráfico de barras
import matplotlib.pyplot as plt 
import matplotlib.ticker as ticker
import pandas as pd # importação das bibiliotecas e modulos

df = pd.read_csv("olist_2016_2018.csv") # leitura do arquivo passando as informações para a variável df
vendas_categorias = df.groupby('product_category_name')['price'].sum().sort_values(ascending=True).tail(10) # soma o valor das 10 categoria com mais vendas

plt.figure(figsize=(10, 6)) # Criação do gráfico
vendas_categorias.plot(kind='barh', color ='skyblue', edgecolor ='black', width=0.8) # tipo de barra e cor


plt.title('Top 10 Categorias com Maior Faturamento', fontsize=16) # título do gráfico
plt.xlabel('Total de Vendas (R$)', fontsize=12) # rótulo eixo x
plt.ylabel('Categoria', fontsize=12) # rótulo eixo y
plt.grid(axis='x', linestyle='--', alpha=0.7) # leves linhas a partir do eixo x para melhor referencial


plt.gca().xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}')) # coverte os números do Total de vendas da notação 1e6 padrão

plt.tight_layout() # evita que parte do gráfico seja cortado

plt.show() # exibe o gráfico

In [ ]:
# Gráfico de linha temporal
import matplotlib.pyplot as plt 
import matplotlib.ticker as ticker
import pandas as pd # importação das bibiliotecas

df = pd.read_csv("olist_2016_2018.csv") # leitura do arquivo passando as informações para a variável df
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp']) # converte para o formato de data
df = df[(df['order_purchase_timestamp'] >= '2017-01-01') & (df['order_purchase_timestamp'] <= '2018-08-31')] # filtra as datas
vendas_mes = df.groupby(df['order_purchase_timestamp'].dt.to_period('M'))['price'].sum() # soma as vendas por mês 

plt.figure(figsize=(10, 6)) # Criação do gráfico
vendas_mes.plot(kind='line', marker='o', color ='darkblue', linewidth=2) # tipo de linha com marcadores


plt.title('Evolução do Faturamento Mensal', fontsize=16) # título do gráfico
plt.xlabel('Mês/Ano', fontsize=12) # rótulo eixo x
plt.ylabel('Total de Vendas (R$)', fontsize=12) # rótulo eixo y

plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}')) # coverte os números do Total de vendas da notação 1e6 padrão

plt.tight_layout() # evita que parte do gráfico seja cortado

plt.show() # exibe o gráfico

In [ ]:
# mapa interativo
import pandas as pd
import plotly.express as px
import numpy as np  # importação das bibliotecas

df = pd.read_csv("olist_2016_2018.csv") # leitura do arquivo
vendas_estados = df.groupby('customer_state')['price'].sum().reset_index() # soma dos valores das vendas

vendas_estados['vendas_log'] = np.log10(vendas_estados['price']) # transforma o valor de vendas em escala logarítima para normalização dos dados

geojson_brasil = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson" # URL pública com a malha dos estados brasileiros

fig = px.choropleth( # Criação do mapa
    vendas_estados,
    geojson=geojson_brasil,
    locations='customer_state',
    featureidkey="properties.sigla",
    color='vendas_log', # usa o log para definir a intensidade da cor
    color_continuous_scale="Blues",
    title="Faturamento Total por Estado (Escala Logarítmica)",
    hover_data={'price': ':,.2f', 'vendas_log': False}, # garante que a legenda mostre o valor original ao passar o mouse sobre os estados
    labels={'price': 'Faturamento (R$)', 'customer_state': 'Estado'}
)

fig.update_geos(fitbounds="locations", visible=False) # enquadra somente o brasil na tela
fig.update_layout(title_font_size=16, title_x=0.5, coloraxis_showscale=True) # centraliza o título

fig.show() # exibe o mapa

In [ ]:
# Treemap interativo
import plotly.express as px
import pandas as pd # importação das bibiliotecas

df = pd.read_csv("olist_2016_2018.csv") # leitura do arquivo passando as informações para a variável df
estados_principais = ['SP', 'RJ', 'MG', 'RS', 'PR']
df_filtrado = df[df['customer_state'].isin(estados_principais)] # cria um filtro com os 5 estados principais
vendas_hierarquia = df_filtrado.groupby(['customer_state', 'product_category_name'])['price'].sum().reset_index() # agrupa somando as vendas por estado e por categoria

fig = px.treemap( # cria um Treemap Hierárquico
    vendas_hierarquia,
    path=['customer_state', 'product_category_name'], # define a hierarquia
    values= 'price', # tamanho dos blocos depende do faturamento
    color= 'price', # cor varia conforme valor
    color_continuous_scale='Blues',
    title='Distribuição Hierárquica de Vendas por Estado e Categoria',
    labels={'price': 'Total de Vendas (R$)', 'customer_state': 'Estado', 'product_category_name': 'Categoria'}  
)

fig.update_layout(title_font_size=16, title_x=0.5) # define tamanho e centraliza o título

fig.show() # exibe o gráfico

In [ ]:
# Grafo dirigido node-link 
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx # importa as bibliotecas

df = pd.read_csv("olist_2016_2018.csv")

conexoes = df.groupby(['seller_state', 'customer_state']).size().reset_index(name='quantidade') # agrupamento das conexões
top_conexoes = conexoes.sort_values(by='quantidade', ascending=False).head(20) # pega os 20 maiores fluxos de venda

G = nx.DiGraph() # criação do grafo dirigido
for _, row in top_conexoes.iterrows(): # percorre cada linha
    G.add_edge(row['seller_state'], row['customer_state'], weight=row['quantidade']) # cria as setas

plt.figure(figsize=(6, 6)) # criação da figura
pos = nx.spring_layout(G, k=30, iterations=100, seed=42) # define posição dos nós

tamanho_no = 1200 # tamanho de cada nó 

nx.draw_networkx_nodes(G, pos, node_size=tamanho_no, node_color='skyblue', edgecolors='black') # customiza aparência do nós
nx.draw_networkx_edges( # Aparência das arestas
    G, pos, 
    arrowstyle='-|>',       
    arrowsize=20,          
    edge_color='gray', 
    width=2.0,
    node_size=tamanho_no   
)

nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold') # rótulo dos nós

plt.title('Grafo Node-Link: Principais Fluxos Comerciais entre Estados', fontsize=16) # título
plt.axis('off') # deativa eixos

plt.tight_layout() # evita que parte do gráfico seja cortado
plt.show() # exibe o grafo